<a href="https://colab.research.google.com/github/BenziJPCorrales/DATA-433-Indiana-Housing-Prediction/blob/main/homemodel_pkl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Load and Filter Data
df = pd.read_csv('indiana_real_estate_2026.csv')
df_sf = df[df['type'] == 'single_family'].copy()
df_sf = df_sf[(df_sf['listPrice'] >= 50000) & (df_sf['listPrice'] <= 1500000)]

# Preprocessing: Fill missing values with median
num_cols = ['sqft', 'stories', 'beds', 'baths', 'garage', 'year_built']
for col in num_cols:
    df_sf[col] = df_sf[col].fillna(df_sf[col].median())

# Modeling
X = df_sf[num_cols]
y = df_sf['listPrice']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Evaluation
predictions = rf_model.predict(X_test)
print(f"R-squared: {r2_score(y_test, predictions):.4f}")

R-squared: 0.6633


In [2]:
import joblib

model_data = {
    "model": rf_model,
    "medians": df_sf[['sqft', 'stories', 'beds', 'baths', 'garage', 'year_built']].median().to_dict()
}
joblib.dump(model_data, 'homemodel.pkl')

['homemodel.pkl']

In [3]:
!pip install -q gradio

In [6]:
import gradio as gr
import joblib
import pandas as pd

data = joblib.load('homemodel.pkl')
model = data['model']
medians = data['medians']

# Prediction function
def predict_price(sqft, stories, beds, baths, garage, year_built):
    input_data = pd.DataFrame(
        [[sqft, stories, beds, baths, garage, year_built]],
        columns=['sqft', 'stories', 'beds', 'baths', 'garage', 'year_built']
    )

    prediction = model.predict(input_data)[0]
    return f"${prediction:,.2f}"

# Custom CSS
custom_css = """
body {
    background-color: #001f4d;
}

.gradio-container {
    background-color: #001f4d !important;
}

/* Title + text */
h1, h2, h3, p {
    color: #FFD700 !important;
    text-align: center;
}

/* Button */
button {
    background-color: #FFD700 !important;
    color: black !important;
    border: none !important;
    font-weight: bold !important;
}

button:hover {
    background-color: #ffcc00 !important;
}

/* Labels */
label {
    color: #FFD700 !important;
}

/* Text inputs */
textarea, input {
    background-color: white !important;
    color: black !important;
}

/* ===== SLIDERS ===== */

/* Slider track (the line) */
.gradio-slider input[type="range"]::-webkit-slider-runnable-track {
    background: #FFD700 !important;
}

/* Slider thumb (circle you drag) */
.gradio-slider input[type="range"]::-webkit-slider-thumb {
    background: #FFD700 !important;
}

/* Firefox support */
input[type="range"]::-moz-range-track {
    background: #FFD700 !important;
}

input[type="range"]::-moz-range-thumb {
    background: #FFD700 !important;
}
"""

# Gradio Interface
with gr.Blocks(
    title="Indiana Home Price Predictor",
    css=custom_css
) as demo:

    gr.Markdown(
        """
        # Indiana 2026 Home Price Estimator
        """
    )

    gr.Markdown(
        """
        ### Adjust the parameters below to see the estimated listing price based on our machine learning model.
        """
    )

    with gr.Row():
        with gr.Column():
            sqft = gr.Number(
                label="Square Footage",
                value=int(medians['sqft']),
                step=100
            )

            year_built = gr.Number(
                label="Year Built",
                value=int(medians['year_built']),
                step=1
            )

            stories = gr.Dropdown(
                choices=[1.0, 1.5, 2.0, 3.0],
                label="Stories",
                value=1.0
            )

        with gr.Column():
            beds = gr.Slider(
                minimum=1,
                maximum=10,
                value=int(medians['beds']),
                step=1,
                label="Bedrooms"
            )

            baths = gr.Slider(
                minimum=1,
                maximum=10,
                value=medians['baths'],
                step=0.5,
                label="Bathrooms"
            )

            garage = gr.Number(
                label="Garage Capacity (Cars)",
                value=int(medians['garage']),
                step=1
            )

    output = gr.Textbox(label="Estimated List Price")

    btn = gr.Button(
        "Calculate Value",
        variant="primary"
    )

    btn.click(
        fn=predict_price,
        inputs=[sqft, stories, beds, baths, garage, year_built],
        outputs=output
    )

demo.launch(share=True)

/tmp/ipykernel_54861/2583583396.py:81: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9df28994143300e1ef.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
